# PS S6E4 | XGB (multi-seed) + CatBoost + LightGBM | 10-Fold | Dual T4 GPU

## 1 · Imports & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
import gc
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgbm
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

# -- Class ordering -----------------------------------------------------------
TARGET         = "Irrigation_Need"
INTERNAL_ORDER = ["Low", "Medium", "High"]   # Training label order
PUBLIC_ORDER   = ["High", "Low", "Medium"]   # Submission column order (alphabetical)

TARGET_INT_MAP = {l: i for i, l in enumerate(INTERNAL_ORDER)}
INT_TARGET_MAP = {i: l for l, i in TARGET_INT_MAP.items()}

# -- Feature lists ------------------------------------------------------------
NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]
ALL_SOURCE = NUMS + CATS

# -- CV / blending hyperparams ------------------------------------------------
N_FOLDS        = 10
SEED           = 42
XGB_SEEDS      = [42, 43]   # Multi-seed ensemble - free score boost
ORIG_ROW_WEIGHT = 0.35      # Down-weight original rows vs synthetic

## 2 · Load Data

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
ORIG_DIR = Path("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset")

train      = pd.read_csv(DATA_DIR / "train.csv").set_index("id")
test       = pd.read_csv(DATA_DIR / "test.csv").set_index("id")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

try:
    orig = pd.read_csv(next(ORIG_DIR.rglob("*.csv")))
    if "id" in orig.columns:
        orig = orig.drop(columns=["id"])
    orig = orig.set_index(pd.RangeIndex(len(orig)))
    print(f"✅ Original dataset loaded: {orig.shape}")
except StopIteration:
    orig = pd.DataFrame(columns=train.columns)
    print("⚠️  Original dataset not found - proceeding with synthetic only")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")

# Encode target to integers (Low=0, Medium=1, High=2)
train[TARGET] = train[TARGET].map(TARGET_INT_MAP)
orig[TARGET]  = orig[TARGET].map(TARGET_INT_MAP)

y_internal = train[TARGET].to_numpy(dtype=np.int64)

# Public-order mapping (for submission alignment)
public_map = {l: i for i, l in enumerate(PUBLIC_ORDER)}
i2p        = {TARGET_INT_MAP[l]: public_map[l] for l in INTERNAL_ORDER}
y_public   = np.array([i2p[v] for v in y_internal], dtype=np.int64)

print(f"\nClass distribution (internal):")
print(pd.Series(y_internal).value_counts().rename(INT_TARGET_MAP).to_string())

## 3 · Feature Engineering

Three complementary layers of features:
1. **Domain physics features** - physically motivated ratios / interactions
2. **Pairwise string combos** - all 19×18/2 = 171 candidate pairs, deduplicated by cardinality
3. **Orig-dataset target priors** - global mean target per category/value from the original CSV


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════

# -- 3a. Factorize categorical columns consistently across splits --------------
def factorize_joint(tr, te, og):
    """Factorize CATS using a shared encoding so train/test/orig are aligned."""
    for col in CATS:
        combined = pd.concat([tr[col], te[col], og[col]], ignore_index=True)
        codes, _ = pd.factorize(combined)
        tr[col] = pd.Series(codes[:len(tr)], index=tr.index).astype("int32").astype("category")
        te[col] = pd.Series(codes[len(tr):len(tr)+len(te)], index=te.index).astype("int32").astype("category")
        og[col] = pd.Series(codes[len(tr)+len(te):], index=og.index).astype("int32").astype("category")

factorize_joint(train, test, orig)
print("Categorical factorization done.")

# -- 3b. Domain physics features ----------------------------------------------
def add_domain_features(*dfs):
    """
    Physics-inspired features from EDA insights:
    - Evapotranspiration proxy (key predictor of crop water demand)
    - Water deficit / supply balance
    - Soil quality index
    - Heat stress / drying force composites
    """
    for d in dfs:
        # Core ETP and water balance
        d["ET_proxy"]       = (d["Temperature_C"] * d["Wind_Speed_kmh"] * d["Sunlight_Hours"]) / (d["Humidity"] + 1)
        d["heat_stress"]    = d["Temperature_C"] * d["Sunlight_Hours"]
        d["drying_force"]   = d["Wind_Speed_kmh"] * d["Temperature_C"] / (d["Humidity"] + 1)
        d["water_supply"]   = d["Rainfall_mm"] + d["Previous_Irrigation_mm"]
        d["water_deficit"]  = d["Soil_Moisture"] - d["water_supply"] * 0.1
        d["holding_cap"]    = d["Soil_Moisture"] * d["Organic_Carbon"]
        d["soil_quality"]   = d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)

        # Ratios - normalise soil moisture against atmospheric drivers
        d["moist_rain"]     = d["Soil_Moisture"] / (d["Rainfall_mm"] + 1)
        d["moist_temp"]     = d["Soil_Moisture"] / (d["Temperature_C"] + 1)
        d["moist_wind"]     = d["Soil_Moisture"] / (d["Wind_Speed_kmh"] + 1)

        # Product interactions
        d["moist_x_wind"]   = d["Soil_Moisture"] * d["Wind_Speed_kmh"]
        d["moist_x_temp"]   = d["Soil_Moisture"] * d["Temperature_C"]
        d["wind_x_temp"]    = d["Wind_Speed_kmh"] * d["Temperature_C"]

        # Polynomial terms (key numerical drivers)
        d["moisture_sq"]    = d["Soil_Moisture"] ** 2
        d["wind_sq"]        = d["Wind_Speed_kmh"] ** 2
        d["temp_sq"]        = d["Temperature_C"] ** 2

        # Log transforms (EDA: right-skewed distributions)
        d["log_rainfall"]   = np.log1p(d["Rainfall_mm"])
        d["log_prev_irrig"] = np.log1p(d["Previous_Irrigation_mm"])
        d["log_field_area"] = np.log1p(d["Field_Area_hectare"])

add_domain_features(train, test, orig)
print("Domain features added.")

# -- 3c. Global orig-dataset target priors -------------------------------------
# These are computed ONCE from the full original dataset - no fold leakage,
# as we are not conditioning on any synthetic train labels.
def add_orig_priors(tr, te, og):
    """
    For each source column (cat + num), compute mean(target) in the ORIGINAL
    dataset and merge it as a numeric prior. Fills missing with 0.5 (uniform).
    """
    created = []
    if og.empty:
        return created
    for col in ALL_SOURCE:
        mapping  = og.groupby(col, observed=False)[TARGET].mean().astype("float32")
        feat_nm  = f"TE_ORIG_{col}"
        tr[feat_nm] = tr[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        te[feat_nm] = te[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        og[feat_nm] = og[col].astype(object).map(mapping).fillna(0.5).astype("float32")
        created.append(feat_nm)
    return created

orig_prior_cols = add_orig_priors(train, test, orig)
print(f"Orig-dataset priors added: {len(orig_prior_cols)} features.")

# -- 3d. Pairwise string combos ------------------------------------------------
# Build ALL 19×18/2 pairwise concatenations of every source feature,
# then drop pairs whose cardinality > 50% of total rows (uninformative).
def build_pair_columns(tr, te, og):
    """
    Generates pairwise string combinations of ALL_SOURCE.
    Factorizes jointly to avoid train/val leakage in ordinal encoding.
    Returns list of kept column names (to be TE-encoded per fold).
    """
    te_cols = []
    total   = len(tr) + len(te) + len(og)
    thresh  = total // 2

    for left, right in combinations(ALL_SOURCE, 2):
        name     = f"{left}__{right}"
        tr[name] = tr[left].astype(str) + "_" + tr[right].astype(str)
        te[name] = te[left].astype(str) + "_" + te[right].astype(str)
        og[name] = og[left].astype(str) + "_" + og[right].astype(str)

        combined   = pd.concat([tr[name], te[name], og[name]], ignore_index=True)
        codes, _   = pd.factorize(combined)
        n_unique   = pd.Series(codes).nunique()

        if n_unique > thresh:
            # Too high cardinality - drop
            tr.drop(columns=[name], inplace=True)
            te.drop(columns=[name], inplace=True)
            og.drop(columns=[name], inplace=True)
            continue

        tr[name] = codes[:len(tr)].astype("int32")
        te[name] = codes[len(tr):len(tr)+len(te)].astype("int32")
        og[name] = codes[len(tr)+len(te):].astype("int32")
        te_cols.append(name)

    return te_cols

print("Building pairwise combo columns (this may take ~1–2 minutes)...")
te_columns = build_pair_columns(train, test, orig)
print(f"Pairwise combos kept : {len(te_columns)}")

# -- Final feature list --------------------------------------------------------
feature_columns = [c for c in train.columns if c != TARGET]
print(f"Total features       : {len(feature_columns)}")

## 4 · Per-Fold Frame Builder

**Key leakage-prevention detail:** `TargetEncoder` is `fit_transform`-ed on `(train_fold ∪ orig)` inside each fold. Only `.transform()` is called on the validation and test sets.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. PER-FOLD FRAME BUILDER (LEAKAGE-FREE)
# ═══════════════════════════════════════════════════════════════════════════════

def build_fold_frames(tr_fold, va_fold, te_df, og_df, feat_cols, te_cols, y_tr, seed):
    """
    Assembles leakage-free train / val / test matrices for one fold.

    Steps
    -----
    1. Combine tr_fold + orig for training (orig rows get sample weight 0.35).
    2. Fit TargetEncoder on the COMBINED (train_fold ∪ orig) set.
    3. Transform val and test separately.
    4. Restore CATS as category dtype for XGBoost / LightGBM.
    5. Return sample weights (balanced × orig downweight).
    """
    base_cols = [c for c in feat_cols if c not in te_cols]

    # Merge train fold with orig for TE fitting
    og_y      = og_df[TARGET].to_numpy(dtype=np.int64)
    combined_X = pd.concat([tr_fold[te_cols], og_df[te_cols]], ignore_index=True)
    combined_y = np.concatenate([y_tr, og_y])

    encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed, smooth="auto")

    def _rename(arr):
        return pd.DataFrame(arr, columns=[f"te_{i}" for i in range(arr.shape[1])])

    enc_tr = _rename(encoder.fit_transform(combined_X, combined_y))
    enc_va = _rename(encoder.transform(va_fold[te_cols]))
    enc_te = _rename(encoder.transform(te_df[te_cols]))

    # Full training matrix = encoded pairs + non-pair base features
    combined_base = pd.concat([tr_fold[base_cols], og_df[base_cols]], ignore_index=True)
    x_tr = pd.concat([enc_tr, combined_base.reset_index(drop=True)], axis=1)
    x_va = pd.concat([enc_va, va_fold[base_cols].reset_index(drop=True)], axis=1)
    x_te = pd.concat([enc_te, te_df[base_cols].reset_index(drop=True)], axis=1)

    # Sample weights: class balance × orig downweight
    sw = compute_sample_weight("balanced", combined_y).astype(np.float32)
    sw[len(tr_fold):] *= ORIG_ROW_WEIGHT   # Down-weight orig rows

    # Restore CATS as category dtype (XGBoost & LightGBM need this)
    cat_base = [c for c in CATS if c in x_tr.columns]
    for df_ in (x_tr, x_va, x_te):
        for c in cat_base:
            df_[c] = df_[c].astype("category")

    return x_tr, combined_y, x_va, x_te, cat_base, sw

## 5 · Custom Metrics & Ensemble Utilities

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5. CUSTOM METRICS & ENSEMBLE UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def bal_acc_xgb_metric():
    """Custom eval metric for XGBoost early stopping on Balanced Accuracy."""
    def _m(y_true, y_pred):
        return balanced_accuracy_score(
            y_true.astype(int), np.argmax(y_pred.reshape(-1, 3), axis=1)
        )
    _m.__name__ = "bal_acc"
    return _m


def reorder_to_public(proba):
    """
    Reorder probability columns from INTERNAL_ORDER to PUBLIC_ORDER.
    INTERNAL: [Low, Medium, High] -> PUBLIC: [High, Low, Medium]
    """
    src = {l: i for i, l in enumerate(INTERNAL_ORDER)}
    return proba[:, [src[l] for l in PUBLIC_ORDER]]


# -- Log-space bias tuner (reference method - most robust) ---------------------
def public_preds(proba, bias):
    """Apply additive log-space bias before argmax (equiv. to class prior shift)."""
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)


def tune_bias_log(proba, y_true):
    """
    Multi-step grid climb in log-probability space.
    Explores decreasing step sizes: 1.0 -> 0.005.
    Returns (best_bias, best_score).
    """
    best_bias  = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-8:
                        best_bias, best_score, improved = c, s, True
    return best_bias, best_score


# -- Nelder-Mead class-weight optimizer (scipy) --------------------------------
def nelder_mead_optimize(proba, y_true):
    """
    Optimises per-class multiplicative weights via Nelder-Mead.
    Complements the log-space bias by scaling the probability surface.
    """
    def neg_ba(weights):
        return -balanced_accuracy_score(y_true, np.argmax(proba * np.abs(weights), axis=1))

    res = minimize(neg_ba, x0=[1.0] * proba.shape[1], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 5000})
    best_weights = np.abs(res.x)
    best_score   = -res.fun
    return best_weights, best_score


# -- 3-way grid search for blend weights --------------------------------------
def search_blend_weights(o_xgb, o_cb, o_lgb, y_true, step=0.05):
    """
    Grid search over (w_xgb, w_cb, w_lgb) s.t. w1+w2+w3=1.
    Returns (best_weights_tuple, best_score).
    """
    best_sc, best_w = 0.0, (1/3, 1/3, 1/3)
    for w1 in np.arange(0.10, 0.80, step):
        for w2 in np.arange(0.05, 0.80 - w1, step):
            w3 = round(1.0 - w1 - w2, 6)
            if w3 < 0.05:
                continue
            bl = w1 * o_xgb + w2 * o_cb + w3 * o_lgb
            s  = balanced_accuracy_score(y_true, bl.argmax(1))
            if s > best_sc + 1e-8:
                best_sc = s
                best_w  = (round(w1, 2), round(w2, 2), round(w3, 2))
    return best_w, best_sc

## 6 · 10-Fold OOF Training Loop

Model configurations:
- **XGBoost** - `device='cuda'`, LR=0.02, max_depth=6, max_bin=1024, 10 000 estimators w/ early stopping=300, run with **2 seeds** per fold
- **CatBoost** - `devices='0:1'` (dual T4!), depth=8, LR=0.04, 2 000 iterations w/ early stopping
- **LightGBM** - num_leaves=127, LR=0.04, 3 000 estimators w/ early stopping=150


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. 10-FOLD OOF TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_xgb = np.zeros((len(train), 3), dtype=np.float64)
oof_cb  = np.zeros((len(train), 3), dtype=np.float64)
oof_lgb = np.zeros((len(train), 3), dtype=np.float64)

tst_xgb_parts = []
tst_cb_parts  = []
tst_lgb_parts = []

print(f"{'='*72}")
print(f"  {N_FOLDS}-FOLD CV  |  XGB × {len(XGB_SEEDS)} seeds  +  CatBoost  +  LightGBM")
print(f"  Total features: {len(feature_columns)}")
print(f"{'='*72}\n")

for fold, (ti, vi) in enumerate(splitter.split(np.zeros(len(train)), y_internal), 1):
    print(f"-- Fold {fold:>2}/{N_FOLDS} ------------------------------------------")

    tr_f, va_f = train.iloc[ti].copy(), train.iloc[vi].copy()
    y_tr = tr_f[TARGET].to_numpy(dtype=np.int64)
    y_va = va_f[TARGET].to_numpy(dtype=np.int64)

    # Build leakage-free matrices
    x_tr, y_m, x_va, x_te, cat_b, sw = build_fold_frames(
        tr_f, va_f, test, orig, feature_columns, te_columns, y_tr, SEED + fold
    )
    cat_idx_cb = [x_tr.columns.get_loc(c) for c in cat_b]

    # -- XGBoost - multi-seed ---------------------------------------------------
    va_xgb_seeds, te_xgb_seeds = [], []
    for s in XGB_SEEDS:
        m_xgb = xgb.XGBClassifier(
            objective         = "multi:softprob",
            num_class         = 3,
            tree_method       = "hist",
            device            = "cuda",
            enable_categorical= True,
            n_jobs            = -1,
            # Tuned hyperparameters
            n_estimators      = 10_000,
            learning_rate     = 0.02,
            max_depth         = 6,
            min_child_weight  = 3,
            subsample         = 0.80,
            colsample_bytree  = 0.80,
            max_bin           = 1024,
            reg_alpha         = 0.1,
            reg_lambda        = 1.0,
            # Metric & early stopping
            eval_metric       = bal_acc_xgb_metric(),
            callbacks         = [xgb.callback.EarlyStopping(
                                    rounds=300, metric_name="bal_acc",
                                    maximize=True, save_best=True)],
            random_state      = s,
        )
        m_xgb.fit(
            x_tr, y_m,
            eval_set      = [(x_va, y_va)],
            sample_weight = sw,
            verbose       = False,
        )
        va_xgb_seeds.append(m_xgb.predict_proba(x_va))
        te_xgb_seeds.append(m_xgb.predict_proba(x_te))
        del m_xgb; gc.collect()

    va_xgb = np.mean(va_xgb_seeds, axis=0)
    te_xgb = np.mean(te_xgb_seeds, axis=0)
    oof_xgb[vi] = va_xgb
    tst_xgb_parts.append(te_xgb.astype(np.float32))

    # -- CatBoost - dual T4 ----------------------------------------------------
    m_cb = CatBoostClassifier(
        loss_function         = "MultiClass",
        eval_metric           = "MultiClass",
        auto_class_weights    = "Balanced",
        iterations            = 2_000,
        learning_rate         = 0.04,
        depth                 = 8,
        l2_leaf_reg           = 4.0,
        random_strength       = 1.0,
        bagging_temperature   = 0.5,
        task_type             = "GPU",
        devices               = "0:1",    # <- Uses BOTH Kaggle T4 GPUs
        random_seed           = SEED + fold,
        verbose               = False,
        early_stopping_rounds = 200,
    )
    m_cb.fit(
        Pool(x_tr, y_m, cat_features=cat_idx_cb, weight=sw),
        eval_set    = Pool(x_va, y_va, cat_features=cat_idx_cb),
        use_best_model = True,
    )
    oof_cb[vi] = m_cb.predict_proba(Pool(x_va, cat_features=cat_idx_cb))
    tst_cb_parts.append(
        m_cb.predict_proba(Pool(x_te, cat_features=cat_idx_cb)).astype(np.float32)
    )
    del m_cb; gc.collect()

    # -- LightGBM --------------------------------------------------------------
    m_lgb = lgbm.LGBMClassifier(
        objective         = "multiclass",
        num_class         = 3,
        metric            = "multi_logloss",
        class_weight      = "balanced",
        n_jobs            = -1,
        verbose           = -1,
        # Tuned hyperparameters
        n_estimators      = 3_000,
        learning_rate     = 0.04,
        num_leaves        = 127,
        min_child_samples = 50,
        subsample         = 0.80,
        colsample_bytree  = 0.60,
        reg_alpha         = 0.10,
        reg_lambda        = 1.00,
        random_state      = SEED + fold,
    )
    m_lgb.fit(
        x_tr, y_m,
        sample_weight = sw,
        eval_set      = [(x_va, y_va)],
        callbacks     = [
            lgbm.early_stopping(150, verbose=False),
            lgbm.log_evaluation(0),
        ],
    )
    oof_lgb[vi] = m_lgb.predict_proba(x_va)
    tst_lgb_parts.append(m_lgb.predict_proba(x_te).astype(np.float32))
    del m_lgb; gc.collect()

    # -- Fold summary ----------------------------------------------------------
    ba_x = balanced_accuracy_score(y_va, va_xgb.argmax(1))
    ba_c = balanced_accuracy_score(y_va, oof_cb[vi].argmax(1))
    ba_l = balanced_accuracy_score(y_va, oof_lgb[vi].argmax(1))
    print(f"  XGB={ba_x:.5f}  CB={ba_c:.5f}  LGB={ba_l:.5f}")

    del x_tr, x_va, x_te, tr_f, va_f
    gc.collect()

# Average test probabilities across folds
mean_tst_xgb = np.mean(np.stack(tst_xgb_parts), axis=0).astype(np.float64)
mean_tst_cb  = np.mean(np.stack(tst_cb_parts),  axis=0).astype(np.float64)
mean_tst_lgb = np.mean(np.stack(tst_lgb_parts), axis=0).astype(np.float64)

print(f"\n{'='*72}")
print("  INDIVIDUAL OOF SCORES (Balanced Accuracy)")
print(f"{'='*72}")
for nm, o in [("XGB", oof_xgb), ("CB", oof_cb), ("LGB", oof_lgb)]:
    score = balanced_accuracy_score(y_internal, o.argmax(1))
    print(f"  {nm:>3} : {score:.5f}")

## 7 · 3-Way Blend Weight Search

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7. 3-WAY BLEND WEIGHT SEARCH
# ═══════════════════════════════════════════════════════════════════════════════
print("Searching optimal blend weights (XGB, CB, LGB)...")
best_w, best_blend_score = search_blend_weights(oof_xgb, oof_cb, oof_lgb, y_internal, step=0.05)

print(f"  Best weights  : XGB={best_w[0]}, CB={best_w[1]}, LGB={best_w[2]}")
print(f"  Blended OOF BA: {best_blend_score:.5f}")

# Build blended OOF & test probabilities
blend_oof  = best_w[0]*oof_xgb + best_w[1]*oof_cb + best_w[2]*oof_lgb
blend_test = best_w[0]*mean_tst_xgb + best_w[1]*mean_tst_cb + best_w[2]*mean_tst_lgb

## 8 · Level-2 Logistic Regression Meta-Stacker

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8. LEVEL-2 LOGISTIC REGRESSION META-STACKER
# ═══════════════════════════════════════════════════════════════════════════════
# Stack all 9 OOF probability columns (3 models × 3 classes) as meta-features.
meta_X_tr = np.hstack([oof_xgb, oof_cb, oof_lgb])
meta_X_te = np.hstack([mean_tst_xgb, mean_tst_cb, mean_tst_lgb])

meta_clf = LogisticRegression(
    class_weight = "balanced",
    max_iter     = 2_000,
    C            = 1.0,
    random_state = SEED,
)
meta_clf.fit(meta_X_tr, y_internal)

stack_oof_proba  = meta_clf.predict_proba(meta_X_tr)
stack_test_proba = meta_clf.predict_proba(meta_X_te)

stack_oof_score  = balanced_accuracy_score(y_internal, stack_oof_proba.argmax(1))
print(f"  Level-2 Stacked OOF BA : {stack_oof_score:.5f}")

# Choose the best base: blended or stacked
if stack_oof_score >= best_blend_score:
    print("  -> Using STACKED probabilities as base for threshold tuning.")
    base_oof_proba  = stack_oof_proba
    base_test_proba = stack_test_proba
    base_label      = "Stack"
else:
    print("  -> Blended probabilities outperform stacker - using BLEND as base.")
    base_oof_proba  = blend_oof
    base_test_proba = blend_test
    base_label      = "Blend"

## 9 · Threshold Optimization

**Two complementary methods, both applied:**
1. **Log-space bias tuning** - additive shift in log(p) space; interpretable as a class prior adjustment
2. **Nelder-Mead** - multiplicative weights on raw probabilities; finer surface exploration

The final submission uses whichever gives the higher OOF score.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9. THRESHOLD OPTIMIZATION (Dual Method)
# ═══════════════════════════════════════════════════════════════════════════════

# -- Reorder columns to PUBLIC order before bias tuning ------------------------
oof_pub  = reorder_to_public(base_oof_proba)
test_pub = reorder_to_public(base_test_proba)

# Method A: Log-space bias tuning
print("Method A - Log-space bias tuning...")
bias_A, score_A = tune_bias_log(oof_pub, y_public)
print(f"  Bias vector : {np.round(bias_A, 4)}")
print(f"  OOF BA      : {score_A:.5f}")

# Method B: Nelder-Mead multiplicative weights
print("\nMethod B - Nelder-Mead multiplicative weights...")
weights_B, score_B = nelder_mead_optimize(oof_pub, y_public)
print(f"  Weights     : {np.round(weights_B, 4)}")
print(f"  OOF BA      : {score_B:.5f}")

# Select the better method for test inference
if score_A >= score_B:
    print(f"\n✅ Using Method A (log-space bias). OOF BA = {score_A:.5f}")
    final_test_preds = public_preds(test_pub, bias_A)
    final_oof_score  = score_A
else:
    print(f"\n✅ Using Method B (Nelder-Mead). OOF BA = {score_B:.5f}")
    final_test_preds = np.argmax(test_pub * weights_B, axis=1)
    final_oof_score  = score_B

decode = np.array(PUBLIC_ORDER)

print(f"\n{'='*72}")
print(f"  🏆 FINAL OOF BALANCED ACCURACY : {final_oof_score:.5f}")
print(f"  (Base: {base_label})")
print(f"{'='*72}")

## 10 · Save Submission

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10. FINAL SUBMISSION
# ═══════════════════════════════════════════════════════════════════════════════
submission = sample_sub.copy()
submission[TARGET] = decode[final_test_preds]

submission.to_csv("submission.csv", index=False)

print("submission.csv saved.")
print("\nPrediction distribution:")
print(submission[TARGET].value_counts().to_string())
print("\nSample:")
print(submission.head(10).to_string(index=False))

## 11 · Ensemble Diagnostics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11. DIAGNOSTICS & VISUALISATION
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Reconstruct internal-order OOF predictions for confusion matrix
final_oof_internal = base_oof_proba.argmax(1)

# -- 1. Confusion Matrix -------------------------------------------------------
cm = confusion_matrix(y_internal, final_oof_internal)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=INTERNAL_ORDER, yticklabels=INTERNAL_ORDER,
            cbar=False, annot_kws={"size": 12})
axes[0].set_title(f"OOF Confusion Matrix\n({base_label} | BA={final_oof_score:.5f})",
                  fontsize=13, pad=12)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

# -- 2. Class Distribution -----------------------------------------------------
dist_df = pd.DataFrame({
    "True"      : [INT_TARGET_MAP[v] for v in y_internal],
    "Predicted" : [INT_TARGET_MAP[v] for v in final_oof_internal],
})
dist_m = dist_df.melt(var_name="Source", value_name="Class")
sns.countplot(data=dist_m, x="Class", hue="Source", ax=axes[1],
              order=INTERNAL_ORDER, palette=["#4C72B0", "#DD8452"])
axes[1].set_title("Class Distribution: True vs OOF Predicted", fontsize=13, pad=12)
axes[1].set_xlabel("Irrigation Need"); axes[1].set_ylabel("Count")
axes[1].legend(title="")

# -- 3. Base Model OOF Correlation ---------------------------------------------
corr_df = pd.DataFrame({
    "XGB_Low" : oof_xgb[:, 0], "XGB_Med" : oof_xgb[:, 1], "XGB_High": oof_xgb[:, 2],
    "CB_Low"  : oof_cb[:, 0],  "CB_Med"  : oof_cb[:, 1],  "CB_High" : oof_cb[:, 2],
    "LGB_Low" : oof_lgb[:, 0], "LGB_Med" : oof_lgb[:, 1], "LGB_High": oof_lgb[:, 2],
})
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt=".3f",
            cmap="coolwarm", vmin=0.7, vmax=1.0, ax=axes[2],
            cbar_kws={"shrink": 0.8})
axes[2].set_title("Base Model OOF Correlation\n(lower = more diverse = better ensemble)",
                  fontsize=13, pad=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("diagnostics.png", dpi=120, bbox_inches="tight")
plt.show()

# -- Save OOF arrays for offline analysis / re-blending -----------------------
np.save("oof_xgb.npy", oof_xgb)
np.save("oof_cb.npy",  oof_cb)
np.save("oof_lgb.npy", oof_lgb)
np.save("test_blend.npy", test_pub)
print("OOF arrays saved: oof_xgb.npy, oof_cb.npy, oof_lgb.npy, test_blend.npy")